## Center-crop training GeoTIFFs to a smaller pixel size

Same layout as **`embed.ipynb`** / **`embed_cls_patch.load_dataset`**: `data_dir / <source_stem> / {split} / {0|1} / *.tif`.

- **Default:** 433×433 inputs → central **32×32** crop (integer offsets like `embed.ipynb` `center_crop_hwc`; off by up to half a pixel vs. a fractional center is fine for training).
- **Georeferencing:** crop uses `rasterio.windows.Window` + `src.window_transform(window)` so the saved GeoTIFF has the correct **affine** for the crop footprint.
- **Output name:** `name.tif` → `name_32px.tif` (suffix encodes crop size when you change `OUT_PX`).
- **Values:** raw band data are copied (no ÷10000); profile metadata follows the source where possible.

Set **`SRC_DATA_DIR`** (existing `training_patches…` tree) and **`DST_DATA_DIR`** (new top-level folder; structure is recreated underneath).

**Symlinks:** you can pass **absolute** paths that are symlinks (e.g. a mount alias). Roots use **`expanduser` + `absolute()`** so symlinks are **not** followed by default. Set **`RESOLVE_SYMLINKS = True`** in the config cell if you want canonical paths via **`.resolve()`**.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import rasterio
from rasterio.windows import Window

parent_dir = Path.cwd().parent
if str(parent_dir) not in sys.path:
    sys.path.insert(0, str(parent_dir))

%load_ext autoreload
%autoreload 2

# --- edit these ---
data_timestamp = "2026-04-08T19:10"
SRC_DATA_DIR = Path(f"../data/training_patches{data_timestamp}")
DST_DATA_DIR = Path(f"../data/training_patches{data_timestamp}_32px")

EXPECTED_IN_PX = 433   # informational; set None to skip size check
OUT_PX = 32

# None = discover splits from disk (like embed.ipynb); or e.g. ["train", "val"]
SPLITS = None

DRY_RUN = False   # if True, only print what would be written
OVERWRITE = False

# Absolute symlink-friendly roots: False = expanduser + absolute() (keep symlinks); True = .resolve()
RESOLVE_SYMLINKS = True

In [ ]:
def training_data_root(path: Path, *, resolve_symlinks: bool = False) -> Path:
    """Normalize SRC/DST: expand ~, make absolute. Preserves symlinks unless resolve_symlinks=True."""
    p = Path(path).expanduser()
    if not p.is_absolute():
        p = (Path.cwd() / p).absolute()
    else:
        p = p.absolute()
    if resolve_symlinks:
        p = p.resolve()
    return p


def iter_source_dirs(root: Path):
    """Top-level source folders under training_patches (same as embed_cls_patch.load_dataset)."""
    for child in sorted(p for p in Path(root).iterdir() if p.is_dir()):
        yield child.name, child


def discover_splits(root: Path) -> list[str]:
    found = set()
    for _stem, source_dir in iter_source_dirs(root):
        for tif_path in source_dir.glob("**/[01]/*.tif"):
            if tif_path.parent.name not in {"0", "1"}:
                continue
            found.add(tif_path.parent.parent.name)
    return sorted(found)


def iter_training_tifs(
    data_dir: Path,
    *,
    splits: list[str] | None,
):
    """
    Yield (tif_path, source_stem) matching embed layout:
    data_dir / <source> / <split> / <0|1> / *.tif
    (same iteration order idea as embed_cls_patch.load_dataset).
    """
    data_dir = Path(data_dir)
    use_splits = discover_splits(data_dir) if splits is None else list(splits)
    for split_name in use_splits:
        for source_stem, source_dir in iter_source_dirs(data_dir):
            for class_subdir in ("0", "1"):
                class_dir = source_dir / split_name / class_subdir
                if not class_dir.is_dir():
                    continue
                for tif_path in sorted(class_dir.glob("*.tif")):
                    yield tif_path, source_stem


def center_crop_window(width: int, height: int, crop_w: int, crop_h: int) -> Window:
    """Top-left of central crop; same integer math as embed.ipynb ``center_crop_hwc``."""
    if height < crop_h or width < crop_w:
        raise ValueError(
            f"Cannot center-crop to {crop_w}×{crop_h}: raster is {width}×{height}"
        )
    col_off = (width - crop_w) // 2
    row_off = (height - crop_h) // 2
    return Window(col_off, row_off, crop_w, crop_h)


def dst_path_for_crop(src: Path, src_root: Path, dst_root: Path, out_px: int) -> Path:
    try:
        rel = src.relative_to(src_root)
    except ValueError:
        rel = src.resolve().relative_to(src_root.resolve())
    out_name = f"{src.stem}_{out_px}px.tif"
    return dst_root / rel.parent / out_name


def write_center_crop(
    src_path: Path,
    dst_path: Path,
    *,
    out_px: int,
    expected_in_px: int | None,
    dry_run: bool,
    overwrite: bool,
) -> str:
    """
    Returns status: 'skip_exists' | 'skip_dry' | 'ok' | 'error:...'
    """
    dst_path = Path(dst_path)
    if dst_path.exists() and not overwrite:
        return "skip_exists"

    with rasterio.open(src_path) as src:
        h, w = src.height, src.width
        if expected_in_px is not None and (h != expected_in_px or w != expected_in_px):
            print(
                f"WARN shape {w}×{h} != expected {expected_in_px}×{expected_in_px}: {src_path}"
            )
        window = center_crop_window(w, h, out_px, out_px)
        out_transform = src.window_transform(window)
        data = src.read(window=window)

        profile = src.profile.copy()
        profile.update(
            height=out_px,
            width=out_px,
            transform=out_transform,
        )
        # Source tiles may use large JPEG/COG blocks; disable tiling for tiny outputs.
        if profile.get("tiled"):
            profile["tiled"] = False
            profile.pop("blockxsize", None)
            profile.pop("blockysize", None)

    if dry_run:
        print(f"[dry-run] {src_path} -> {dst_path}")
        return "skip_dry"

    dst_path.parent.mkdir(parents=True, exist_ok=True)
    with rasterio.open(dst_path, "w", **profile) as dst:
        dst.write(data)
    return "ok"

In [ ]:
src_root = training_data_root(SRC_DATA_DIR, resolve_symlinks=RESOLVE_SYMLINKS)
dst_root = training_data_root(DST_DATA_DIR, resolve_symlinks=RESOLVE_SYMLINKS)
splits_eff = discover_splits(src_root) if SPLITS is None else SPLITS
print(f"Source root: {src_root}")
print(f"Dest root:   {dst_root}")
print(f"Splits:      {splits_eff}")
print(f"Crop:        center {OUT_PX}×{OUT_PX}")

counts = {"ok": 0, "skip_exists": 0, "skip_dry": 0, "error": 0}
for tif_path, _source in iter_training_tifs(src_root, splits=SPLITS):
    #if 'v3.6' not in str(tif_path):
    #    continue
    out_path = dst_path_for_crop(tif_path, src_root, dst_root, OUT_PX)
    try:
        st = write_center_crop(
            tif_path,
            out_path,
            out_px=OUT_PX,
            expected_in_px=EXPECTED_IN_PX,
            dry_run=DRY_RUN,
            overwrite=OVERWRITE,
        )
    except Exception as e:
        print(f"ERROR {tif_path}: {e}")
        counts["error"] += 1
        continue
    if st.startswith("error"):
        counts["error"] += 1
    else:
        counts[st] = counts.get(st, 0) + 1

print(counts)

### RGB preview PNGs (one grid per `source_stem`)

Uses **`TrainingData.make_rgb_preview()`** and **`write_thumbnail_grid()`** from **`get_training_data.py`**: reads cropped `*_{OUT_PX}px.tif` chips under **`DST_DATA_DIR`**, converts reflectance (DN÷10000) to uint8 RGB the same way as the GEE pipeline, and writes **`DST_DATA_DIR / "{source_stem}_rgb_preview.png"`** (subsampled to **`PREVIEW_MAX_TILES`** tiles like `fetch_tiles_for_points`).

Run after the crop pass so the `_32px.tif` files exist. Requires notebook cwd **`gee/`** (or adjust `sys.path`) so `get_training_data` imports cleanly.

In [ ]:
# Assumes cells above defined OUT_PX, DST_DATA_DIR and ran the crop step.
_resolve = globals().get("RESOLVE_SYMLINKS", False)

if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

import gee
import numpy as np
from get_training_data import TrainingData, write_thumbnail_grid

PREVIEW_MAX_TILES = 1000  # random subsample if more crops exist (same idea as fetch_tiles_for_points)

dst_root = training_data_root(DST_DATA_DIR, resolve_symlinks=_resolve)
_preview_td = TrainingData(gee.DataConfig())


def _load_hwc_reflectance(tif_path: Path) -> np.ndarray:
    with rasterio.open(tif_path) as src:
        arr = src.read()
    return np.moveaxis(arr, 0, -1).astype(np.float32)


def _iter_source_cropped_tifs(root: Path, out_px: int):
    """Top-level source folders; all **/*_{out_px}px.tif under each."""
    for source_dir in sorted(p for p in Path(root).iterdir() if p.is_dir()):
        pat = f"*_{out_px}px.tif"
        paths = sorted(source_dir.glob(f"**/{pat}"))
        yield source_dir.name, paths


for source_stem, paths in _iter_source_cropped_tifs(dst_root, OUT_PX):
    if not paths:
        print(f"skip {source_stem!r}: no *_{OUT_PX}px.tif under {dst_root / source_stem}")
        continue
    rgb_tiles = []
    for p in paths:
        hwc = _load_hwc_reflectance(p)
        rgb_tiles.append(_preview_td.make_rgb_preview(hwc))
    out_png = dst_root / f"{source_stem}_rgb_preview.png"
    write_thumbnail_grid(
        rgb_tiles, out_png, tilesize=OUT_PX, max_tiles=PREVIEW_MAX_TILES
    )
    print(f"Wrote {out_png} ({len(rgb_tiles)} tiles, grid may subsample to {PREVIEW_MAX_TILES})")